In [6]:
import os
import pandas as pd
import numpy as np

# Función para calcular el Awesome Oscillator
def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Directorio donde se encuentran los archivos CSV
directory_path = 'D:\iqrobot\df_test\\eurgbp-otc\\5m'
# DataFrame para acumular todos los resultados
combined_results = []

# Recorrer todos los archivos CSV en el directorio
for filename in os.listdir(directory_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory_path, filename)
        
        # Cargar los datos desde el archivo CSV
        data = pd.read_csv(file_path, parse_dates=['at'])
        
        # Renombrar columnas
        data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
        data.set_index('Date', inplace=True)

        # Calcular Awesome Oscillator
        data = awesome_oscillator(data)

        # Calcular medias móviles simples (SMA) para SSL Channel
        period = 10
        data['smaHigh'] = data['High'].rolling(window=period).mean()
        data['smaLow'] = data['Low'].rolling(window=period).mean()

        # Inicialización de Hlv
        data['Hlv'] = np.nan

        # Calcular Hlv para SSL Channel
        for i in range(period, len(data)):
            if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
                data['Hlv'].iloc[i] = 1
            elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
                data['Hlv'].iloc[i] = -1
            else:
                data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

        # Calcular sslDown y sslUp para SSL Channel
        data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
        data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

        # Detectar cruces para SSL Channel
        data['cross'] = np.where(
            ((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp'])) |
            ((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp'])),
            1, 0
        )

        # Determinar tipo de cruce para SSL Channel
        data['cross_type'] = np.where(
            (data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
            'Bajista',
            np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                     'Alcista', np.nan)
        )

        # Crear DataFrame para almacenar los resultados de los cruces para SSL Channel
        crosses = data[data['cross'] == 1].copy()
        crosses['close_at_cross'] = crosses['Close']

        # Determinar dirección del cierre de la vela
        data['direccion_vela'] = np.where(data['Close'] > data['Open'], 'Alcista', 'Bajista')

        # Determinar posición del Awesome Oscillator
        data['ao_position'] = np.where(data['ao'] > 0, 'Alcista', 'Bajista')

        # Determinar posición del SSL
        data['ssl_position'] = np.where(data['Hlv'] > 0, 'Alcista', 'Bajista')

        # Crear columna de operación
        data['operacion'] = np.where(
            data['ao_position'] == data['ssl_position'], 
            data['ao_position'],
            'Discrepancia'
        )

        # Calcular el resultado según las condiciones dadas
        data['resultado'] = np.where(
            data['operacion'] == data['direccion_vela'], 
            85,
            np.where(data['operacion'] == 'Discrepancia', 'Discrepancia', -100)
        )

        # Calcular el número de discrepancias y resultados
        discrepancias = data[data['resultado'] == 'Discrepancia']['resultado'].count()
        resultado_85 = data[data['resultado'] == '85']['resultado'].count()
        resultado_100 = data[data['resultado'] == '-100']['resultado'].count()

        # Encontrar ocurrencias consecutivas de 85
        consecutivos_85 = (data['resultado'] == '85').astype(int).groupby((data['resultado'] != '85').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_85 = consecutivos_85.value_counts()

        # Encontrar ocurrencias consecutivas de -100
        consecutivos_100 = (data['resultado'] == '-100').astype(int).groupby((data['resultado'] != '-100').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_100 = consecutivos_100.value_counts()

        # Calcular el total de pérdida y el mínimo a ganar con el profit base
        total_perdida = resultado_100 * 100
        minimo_ganar = resultado_85 * 85

        # Calcular el interés compuesto con capital inicial
        capital_inicial = 100
        tasa_interes = 0.85

        ciclos = [2, 3, 4]
        racha_85 = [veces_consecutivas_85[2] if 2 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[3] if 3 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[4] if 4 in veces_consecutivas_85 else 0]

        racha_100 = [veces_consecutivas_100[2] if 2 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[3] if 3 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[4] if 4 in veces_consecutivas_100 else 0]

        interes_compuesto = []

        for ciclo, factor in zip(ciclos, racha_85):
            capital_final = capital_inicial * (1 + tasa_interes) ** ciclo * factor
            interes_compuesto.append(capital_final)

        # Crear diccionario con los resultados
        resultados = {
            'Archivo': filename,
            'Número de discrepancias': discrepancias,
            'Número de resultados 85': resultado_85,
            'Número de -100': resultado_100,
            'Número de veces que aparece 85 2 veces consecutivas': veces_consecutivas_85.get(2, 0),
            'Número de veces que aparece 85 3 veces consecutivas': veces_consecutivas_85.get(3, 0),
            'Número de veces que aparece 85 4 veces consecutivas': veces_consecutivas_85.get(4, 0),
            'Número de veces que aparece -100 2 veces consecutivas': veces_consecutivas_100.get(2, 0),
            'Número de veces que aparece -100 3 veces consecutivas': veces_consecutivas_100.get(3, 0),
            'Número de veces que aparece -100 4 veces consecutivas': veces_consecutivas_100.get(4, 0),
            'Total de pérdida': total_perdida,
            'Mínimo a ganar con el profit base': minimo_ganar,
            'Interés compuesto a un 85% para 2 ciclos': interes_compuesto[0] if len(interes_compuesto) > 0 else 0,
            'Interés compuesto a un 85% para 3 ciclos': interes_compuesto[1] if len(interes_compuesto) > 1 else 0,
            'Interés compuesto a un 85% para 4 ciclos': interes_compuesto[2] if len(interes_compuesto) > 2 else 0
        }

        # Agregar resultados a la lista
        combined_results.append(resultados)

# Convertir la lista de diccionarios en un DataFrame
combined_results_df = pd.DataFrame(combined_results)

# Guardar los resultados combinados en un archivo CSV
combined_results_df.to_csv(r'D:\\iqrobot\\resultados\\EURGBP-OTCresultados_5m.csv', index=False)


In [7]:
import os
import pandas as pd
import numpy as np

# Función para calcular el Awesome Oscillator
def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Directorio donde se encuentran los archivos CSV
directory_path = 'D:\iqrobot\df_test\\eurgbp-otc\\15m'
# DataFrame para acumular todos los resultados
combined_results = []

# Recorrer todos los archivos CSV en el directorio
for filename in os.listdir(directory_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory_path, filename)
        
        # Cargar los datos desde el archivo CSV
        data = pd.read_csv(file_path, parse_dates=['at'])
        
        # Renombrar columnas
        data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
        data.set_index('Date', inplace=True)

        # Calcular Awesome Oscillator
        data = awesome_oscillator(data)

        # Calcular medias móviles simples (SMA) para SSL Channel
        period = 10
        data['smaHigh'] = data['High'].rolling(window=period).mean()
        data['smaLow'] = data['Low'].rolling(window=period).mean()

        # Inicialización de Hlv
        data['Hlv'] = np.nan

        # Calcular Hlv para SSL Channel
        for i in range(period, len(data)):
            if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
                data['Hlv'].iloc[i] = 1
            elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
                data['Hlv'].iloc[i] = -1
            else:
                data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

        # Calcular sslDown y sslUp para SSL Channel
        data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
        data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

        # Detectar cruces para SSL Channel
        data['cross'] = np.where(
            ((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp'])) |
            ((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp'])),
            1, 0
        )

        # Determinar tipo de cruce para SSL Channel
        data['cross_type'] = np.where(
            (data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
            'Bajista',
            np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                     'Alcista', np.nan)
        )

        # Crear DataFrame para almacenar los resultados de los cruces para SSL Channel
        crosses = data[data['cross'] == 1].copy()
        crosses['close_at_cross'] = crosses['Close']

        # Determinar dirección del cierre de la vela
        data['direccion_vela'] = np.where(data['Close'] > data['Open'], 'Alcista', 'Bajista')

        # Determinar posición del Awesome Oscillator
        data['ao_position'] = np.where(data['ao'] > 0, 'Alcista', 'Bajista')

        # Determinar posición del SSL
        data['ssl_position'] = np.where(data['Hlv'] > 0, 'Alcista', 'Bajista')

        # Crear columna de operación
        data['operacion'] = np.where(
            data['ao_position'] == data['ssl_position'], 
            data['ao_position'],
            'Discrepancia'
        )

        # Calcular el resultado según las condiciones dadas
        data['resultado'] = np.where(
            data['operacion'] == data['direccion_vela'], 
            85,
            np.where(data['operacion'] == 'Discrepancia', 'Discrepancia', -100)
        )

        # Calcular el número de discrepancias y resultados
        discrepancias = data[data['resultado'] == 'Discrepancia']['resultado'].count()
        resultado_85 = data[data['resultado'] == '85']['resultado'].count()
        resultado_100 = data[data['resultado'] == '-100']['resultado'].count()

        # Encontrar ocurrencias consecutivas de 85
        consecutivos_85 = (data['resultado'] == '85').astype(int).groupby((data['resultado'] != '85').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_85 = consecutivos_85.value_counts()

        # Encontrar ocurrencias consecutivas de -100
        consecutivos_100 = (data['resultado'] == '-100').astype(int).groupby((data['resultado'] != '-100').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_100 = consecutivos_100.value_counts()

        # Calcular el total de pérdida y el mínimo a ganar con el profit base
        total_perdida = resultado_100 * 100
        minimo_ganar = resultado_85 * 85

        # Calcular el interés compuesto con capital inicial
        capital_inicial = 100
        tasa_interes = 0.85

        ciclos = [2, 3, 4]
        racha_85 = [veces_consecutivas_85[2] if 2 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[3] if 3 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[4] if 4 in veces_consecutivas_85 else 0]

        racha_100 = [veces_consecutivas_100[2] if 2 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[3] if 3 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[4] if 4 in veces_consecutivas_100 else 0]

        interes_compuesto = []

        for ciclo, factor in zip(ciclos, racha_85):
            capital_final = capital_inicial * (1 + tasa_interes) ** ciclo * factor
            interes_compuesto.append(capital_final)

        # Crear diccionario con los resultados
        resultados = {
            'Archivo': filename,
            'Número de discrepancias': discrepancias,
            'Número de resultados 85': resultado_85,
            'Número de -100': resultado_100,
            'Número de veces que aparece 85 2 veces consecutivas': veces_consecutivas_85.get(2, 0),
            'Número de veces que aparece 85 3 veces consecutivas': veces_consecutivas_85.get(3, 0),
            'Número de veces que aparece 85 4 veces consecutivas': veces_consecutivas_85.get(4, 0),
            'Número de veces que aparece -100 2 veces consecutivas': veces_consecutivas_100.get(2, 0),
            'Número de veces que aparece -100 3 veces consecutivas': veces_consecutivas_100.get(3, 0),
            'Número de veces que aparece -100 4 veces consecutivas': veces_consecutivas_100.get(4, 0),
            'Total de pérdida': total_perdida,
            'Mínimo a ganar con el profit base': minimo_ganar,
            'Interés compuesto a un 85% para 2 ciclos': interes_compuesto[0] if len(interes_compuesto) > 0 else 0,
            'Interés compuesto a un 85% para 3 ciclos': interes_compuesto[1] if len(interes_compuesto) > 1 else 0,
            'Interés compuesto a un 85% para 4 ciclos': interes_compuesto[2] if len(interes_compuesto) > 2 else 0
        }

        # Agregar resultados a la lista
        combined_results.append(resultados)

# Convertir la lista de diccionarios en un DataFrame
combined_results_df = pd.DataFrame(combined_results)

# Guardar los resultados combinados en un archivo CSV
combined_results_df.to_csv(r'D:\\iqrobot\\resultados\\EURGBP-OTCresultados_15m.csv', index=False)


In [8]:
import os
import pandas as pd
import numpy as np

# Función para calcular el Awesome Oscillator
def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Directorio donde se encuentran los archivos CSV
directory_path = 'D:\iqrobot\df_test\\eurgbp-otc\\1m'
# DataFrame para acumular todos los resultados
combined_results = []

# Recorrer todos los archivos CSV en el directorio
for filename in os.listdir(directory_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory_path, filename)
        
        # Cargar los datos desde el archivo CSV
        data = pd.read_csv(file_path, parse_dates=['at'])
        
        # Renombrar columnas
        data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
        data.set_index('Date', inplace=True)

        # Calcular Awesome Oscillator
        data = awesome_oscillator(data)

        # Calcular medias móviles simples (SMA) para SSL Channel
        period = 10
        data['smaHigh'] = data['High'].rolling(window=period).mean()
        data['smaLow'] = data['Low'].rolling(window=period).mean()

        # Inicialización de Hlv
        data['Hlv'] = np.nan

        # Calcular Hlv para SSL Channel
        for i in range(period, len(data)):
            if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
                data['Hlv'].iloc[i] = 1
            elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
                data['Hlv'].iloc[i] = -1
            else:
                data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

        # Calcular sslDown y sslUp para SSL Channel
        data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
        data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

        # Detectar cruces para SSL Channel
        data['cross'] = np.where(
            ((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp'])) |
            ((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp'])),
            1, 0
        )

        # Determinar tipo de cruce para SSL Channel
        data['cross_type'] = np.where(
            (data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
            'Bajista',
            np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                     'Alcista', np.nan)
        )

        # Crear DataFrame para almacenar los resultados de los cruces para SSL Channel
        crosses = data[data['cross'] == 1].copy()
        crosses['close_at_cross'] = crosses['Close']

        # Determinar dirección del cierre de la vela
        data['direccion_vela'] = np.where(data['Close'] > data['Open'], 'Alcista', 'Bajista')

        # Determinar posición del Awesome Oscillator
        data['ao_position'] = np.where(data['ao'] > 0, 'Alcista', 'Bajista')

        # Determinar posición del SSL
        data['ssl_position'] = np.where(data['Hlv'] > 0, 'Alcista', 'Bajista')

        # Crear columna de operación
        data['operacion'] = np.where(
            data['ao_position'] == data['ssl_position'], 
            data['ao_position'],
            'Discrepancia'
        )

        # Calcular el resultado según las condiciones dadas
        data['resultado'] = np.where(
            data['operacion'] == data['direccion_vela'], 
            85,
            np.where(data['operacion'] == 'Discrepancia', 'Discrepancia', -100)
        )

        # Calcular el número de discrepancias y resultados
        discrepancias = data[data['resultado'] == 'Discrepancia']['resultado'].count()
        resultado_85 = data[data['resultado'] == '85']['resultado'].count()
        resultado_100 = data[data['resultado'] == '-100']['resultado'].count()

        # Encontrar ocurrencias consecutivas de 85
        consecutivos_85 = (data['resultado'] == '85').astype(int).groupby((data['resultado'] != '85').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_85 = consecutivos_85.value_counts()

        # Encontrar ocurrencias consecutivas de -100
        consecutivos_100 = (data['resultado'] == '-100').astype(int).groupby((data['resultado'] != '-100').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_100 = consecutivos_100.value_counts()

        # Calcular el total de pérdida y el mínimo a ganar con el profit base
        total_perdida = resultado_100 * 100
        minimo_ganar = resultado_85 * 85

        # Calcular el interés compuesto con capital inicial
        capital_inicial = 100
        tasa_interes = 0.85

        ciclos = [2, 3, 4]
        racha_85 = [veces_consecutivas_85[2] if 2 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[3] if 3 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[4] if 4 in veces_consecutivas_85 else 0]

        racha_100 = [veces_consecutivas_100[2] if 2 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[3] if 3 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[4] if 4 in veces_consecutivas_100 else 0]

        interes_compuesto = []

        for ciclo, factor in zip(ciclos, racha_85):
            capital_final = capital_inicial * (1 + tasa_interes) ** ciclo * factor
            interes_compuesto.append(capital_final)

        # Crear diccionario con los resultados
        resultados = {
            'Archivo': filename,
            'Número de discrepancias': discrepancias,
            'Número de resultados 85': resultado_85,
            'Número de -100': resultado_100,
            'Número de veces que aparece 85 2 veces consecutivas': veces_consecutivas_85.get(2, 0),
            'Número de veces que aparece 85 3 veces consecutivas': veces_consecutivas_85.get(3, 0),
            'Número de veces que aparece 85 4 veces consecutivas': veces_consecutivas_85.get(4, 0),
            'Número de veces que aparece -100 2 veces consecutivas': veces_consecutivas_100.get(2, 0),
            'Número de veces que aparece -100 3 veces consecutivas': veces_consecutivas_100.get(3, 0),
            'Número de veces que aparece -100 4 veces consecutivas': veces_consecutivas_100.get(4, 0),
            'Total de pérdida': total_perdida,
            'Mínimo a ganar con el profit base': minimo_ganar,
            'Interés compuesto a un 85% para 2 ciclos': interes_compuesto[0] if len(interes_compuesto) > 0 else 0,
            'Interés compuesto a un 85% para 3 ciclos': interes_compuesto[1] if len(interes_compuesto) > 1 else 0,
            'Interés compuesto a un 85% para 4 ciclos': interes_compuesto[2] if len(interes_compuesto) > 2 else 0
        }

        # Agregar resultados a la lista
        combined_results.append(resultados)

# Convertir la lista de diccionarios en un DataFrame
combined_results_df = pd.DataFrame(combined_results)

# Guardar los resultados combinados en un archivo CSV
combined_results_df.to_csv(r'D:\\iqrobot\\resultados\\EURGBP-OTCresultados_1m.csv', index=False)


In [9]:
import os
import pandas as pd
import numpy as np

# Función para calcular el Awesome Oscillator
def awesome_oscillator(df, short_period=5, long_period=34):
    df['hl2'] = (df['High'] + df['Low']) / 2
    df['ao'] = df['hl2'].rolling(short_period).mean() - df['hl2'].rolling(long_period).mean()
    df['trend'] = np.where(df['ao'] > 0, 'Alcista', 'Bajista')
    return df

# Directorio donde se encuentran los archivos CSV
directory_path = 'D:\iqrobot\df_test\\eurgbp-otc\\1h'
# DataFrame para acumular todos los resultados
combined_results = []

# Recorrer todos los archivos CSV en el directorio
for filename in os.listdir(directory_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(directory_path, filename)
        
        # Cargar los datos desde el archivo CSV
        data = pd.read_csv(file_path, parse_dates=['at'])
        
        # Renombrar columnas
        data.rename(columns={'at': 'Date', 'open': 'Open', 'max': 'High', 'min': 'Low', 'close': 'Close'}, inplace=True)
        data.set_index('Date', inplace=True)

        # Calcular Awesome Oscillator
        data = awesome_oscillator(data)

        # Calcular medias móviles simples (SMA) para SSL Channel
        period = 10
        data['smaHigh'] = data['High'].rolling(window=period).mean()
        data['smaLow'] = data['Low'].rolling(window=period).mean()

        # Inicialización de Hlv
        data['Hlv'] = np.nan

        # Calcular Hlv para SSL Channel
        for i in range(period, len(data)):
            if data['Close'].iloc[i] > data['smaHigh'].iloc[i]:
                data['Hlv'].iloc[i] = 1
            elif data['Close'].iloc[i] < data['smaLow'].iloc[i]:
                data['Hlv'].iloc[i] = -1
            else:
                data['Hlv'].iloc[i] = data['Hlv'].iloc[i - 1]

        # Calcular sslDown y sslUp para SSL Channel
        data['sslDown'] = np.where(data['Hlv'] < 0, data['smaHigh'], data['smaLow'])
        data['sslUp'] = np.where(data['Hlv'] < 0, data['smaLow'], data['smaHigh'])

        # Detectar cruces para SSL Channel
        data['cross'] = np.where(
            ((data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp'])) |
            ((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp'])),
            1, 0
        )

        # Determinar tipo de cruce para SSL Channel
        data['cross_type'] = np.where(
            (data['sslDown'].shift(1) < data['sslUp'].shift(1)) & (data['sslDown'] > data['sslUp']),
            'Bajista',
            np.where((data['sslDown'].shift(1) > data['sslUp'].shift(1)) & (data['sslDown'] < data['sslUp']),
                     'Alcista', np.nan)
        )

        # Crear DataFrame para almacenar los resultados de los cruces para SSL Channel
        crosses = data[data['cross'] == 1].copy()
        crosses['close_at_cross'] = crosses['Close']

        # Determinar dirección del cierre de la vela
        data['direccion_vela'] = np.where(data['Close'] > data['Open'], 'Alcista', 'Bajista')

        # Determinar posición del Awesome Oscillator
        data['ao_position'] = np.where(data['ao'] > 0, 'Alcista', 'Bajista')

        # Determinar posición del SSL
        data['ssl_position'] = np.where(data['Hlv'] > 0, 'Alcista', 'Bajista')

        # Crear columna de operación
        data['operacion'] = np.where(
            data['ao_position'] == data['ssl_position'], 
            data['ao_position'],
            'Discrepancia'
        )

        # Calcular el resultado según las condiciones dadas
        data['resultado'] = np.where(
            data['operacion'] == data['direccion_vela'], 
            85,
            np.where(data['operacion'] == 'Discrepancia', 'Discrepancia', -100)
        )

        # Calcular el número de discrepancias y resultados
        discrepancias = data[data['resultado'] == 'Discrepancia']['resultado'].count()
        resultado_85 = data[data['resultado'] == '85']['resultado'].count()
        resultado_100 = data[data['resultado'] == '-100']['resultado'].count()

        # Encontrar ocurrencias consecutivas de 85
        consecutivos_85 = (data['resultado'] == '85').astype(int).groupby((data['resultado'] != '85').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_85 = consecutivos_85.value_counts()

        # Encontrar ocurrencias consecutivas de -100
        consecutivos_100 = (data['resultado'] == '-100').astype(int).groupby((data['resultado'] != '-100').cumsum()).cumsum()

        # Contar las veces que aparecen 2, 3 y 4 veces consecutivas
        veces_consecutivas_100 = consecutivos_100.value_counts()

        # Calcular el total de pérdida y el mínimo a ganar con el profit base
        total_perdida = resultado_100 * 100
        minimo_ganar = resultado_85 * 85

        # Calcular el interés compuesto con capital inicial
        capital_inicial = 100
        tasa_interes = 0.85

        ciclos = [2, 3, 4]
        racha_85 = [veces_consecutivas_85[2] if 2 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[3] if 3 in veces_consecutivas_85 else 0,
                    veces_consecutivas_85[4] if 4 in veces_consecutivas_85 else 0]

        racha_100 = [veces_consecutivas_100[2] if 2 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[3] if 3 in veces_consecutivas_100 else 0,
                     veces_consecutivas_100[4] if 4 in veces_consecutivas_100 else 0]

        interes_compuesto = []

        for ciclo, factor in zip(ciclos, racha_85):
            capital_final = capital_inicial * (1 + tasa_interes) ** ciclo * factor
            interes_compuesto.append(capital_final)

        # Crear diccionario con los resultados
        resultados = {
            'Archivo': filename,
            'Número de discrepancias': discrepancias,
            'Número de resultados 85': resultado_85,
            'Número de -100': resultado_100,
            'Número de veces que aparece 85 2 veces consecutivas': veces_consecutivas_85.get(2, 0),
            'Número de veces que aparece 85 3 veces consecutivas': veces_consecutivas_85.get(3, 0),
            'Número de veces que aparece 85 4 veces consecutivas': veces_consecutivas_85.get(4, 0),
            'Número de veces que aparece -100 2 veces consecutivas': veces_consecutivas_100.get(2, 0),
            'Número de veces que aparece -100 3 veces consecutivas': veces_consecutivas_100.get(3, 0),
            'Número de veces que aparece -100 4 veces consecutivas': veces_consecutivas_100.get(4, 0),
            'Total de pérdida': total_perdida,
            'Mínimo a ganar con el profit base': minimo_ganar,
            'Interés compuesto a un 85% para 2 ciclos': interes_compuesto[0] if len(interes_compuesto) > 0 else 0,
            'Interés compuesto a un 85% para 3 ciclos': interes_compuesto[1] if len(interes_compuesto) > 1 else 0,
            'Interés compuesto a un 85% para 4 ciclos': interes_compuesto[2] if len(interes_compuesto) > 2 else 0
        }

        # Agregar resultados a la lista
        combined_results.append(resultados)

# Convertir la lista de diccionarios en un DataFrame
combined_results_df = pd.DataFrame(combined_results)

# Guardar los resultados combinados en un archivo CSV
combined_results_df.to_csv(r'D:\\iqrobot\\resultados\\EURGBP-OTCresultados_1h.csv', index=False)
